# Phase 3 — Fixed model setup and validation

**Study:** Stack Overflow Developer Survey, 2011–2026 · annual salary regression  
**Input:** `combined/all_years_fixed.csv` · **Target:** `Comp_USD_Annual`  
**Output (in memory only):** `df` (unaltered loaded data), `X`, `y`, `salary_outlier_flag`, `region_metadata`  
**Boundary:** This notebook **does not** split the data, fit a transformer, scale values, estimate imputation statistics, or train a model.

Run in order using **Kernel → Restart & Run All**. Run from the project root, where `combined/all_years_fixed.csv` exists. No CSVs or other files are written by this notebook.

### The Phase 3 contract
1. Load the exported 33-column survey dataset while preserving literal answers such as `None` and `NA`.
2. Validate salary and define `y = ln(Comp_USD_Annual)`; do **not** silently filter respondents again.
3. Select only approved model inputs, excluding salary leakage, IDs, target-derived flags, and downstream outcomes.
4. Label missing *categorical* responses as **Not asked** or **No answer** according to a fixed survey-year rule.
5. Keep `survey_year` numeric and keep missing *numeric* values for **training-only** handling in later phases.
6. Run explicit assertions for model schema, missingness, target validity, and `X`/`y` alignment.

## 1. Load source data and verify schema

The import settings are intentional: `keep_default_na=False, na_values=[""]` converts **only blank fields** to missing values. Text answers such as `None` and `NA` remain literal categories. We keep the imported table named `df` unchanged for traceability.

The size from the previous run is shown as a **provenance warning** rather than a hard constraint, because a freshly regenerated CSV may have a legitimate different row count.

In [18]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = Path("combined/all_years_fixed.csv")
TARGET = "Comp_USD_Annual"
MIN_SALARY, MAX_SALARY = 1_000, 1_000_000

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing input: {DATA_PATH.resolve()}. "
        "Launch the notebook from your project root and verify the cleaning export."
    )

df = pd.read_csv(
    DATA_PATH,
    keep_default_na=False,
    na_values=[""],
    low_memory=False,
)

EXPECTED_INPUTS = {
    TARGET, "survey_year", "Country", "Country_grouped", "Region",
    "comp_outlier_country", "Age", "YearsCode", "YearsCodePro", "WorkExp",
    "OrgSize", "EdLevel", "Employment", "Industry", "RemoteWork",
    "LearnCode", "OpSys", "MainBranch", "DevType",
    "LanguageHaveWorkedWith", "LanguageWantToWorkWith",
}
missing_columns = sorted(EXPECTED_INPUTS - set(df.columns))
if missing_columns:
    raise KeyError(f"Export is missing expected fields: {missing_columns}")
if not df.columns.is_unique or not df.index.is_unique:
    raise ValueError("Duplicate columns or row index in exported dataset.")

source_year = pd.to_numeric(df["survey_year"], errors="raise")
if source_year.isna().any() or not (source_year % 1 == 0).all():
    raise ValueError("survey_year must contain integer years without missing values.")
# Do not rewrite the loaded df: type conversion is done in the X copy below.

print("=== SOURCE EXPORT ===")
print("Path:", DATA_PATH.resolve())
print("Rows / columns:", df.shape)
print("Years:", sorted(df["survey_year"].unique().tolist()))
if df.shape != (386_701, 33):
    print("PROVENANCE NOTE: Previous execution used (386701, 33); inspect changes in the cleaning export.")

=== SOURCE EXPORT ===
Path: C:\Users\KWK\Desktop\colege\Project\ECS171-SOF\combined\all_years_fixed.csv
Rows / columns: (386701, 33)
Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


## 2. Validate and log-transform salary

The target is **annual nominal USD** as defined by the *selected v2 cleaning export*, **not** necessarily the separately audited compensation pipeline. The source file already uses the agreed $1,000–$1,000,000 window; Phase 3 only verifies it.

With a log target, a coefficient for a one-unit change in a numeric predictor has exact percentage effect `100 × (exp(β) − 1)%` under the specified linear model, holding other variables fixed. For small coefficients, `100 × β%` is an approximation. This is **association**, not automatically causation.

In [19]:
salary = pd.to_numeric(df[TARGET], errors="raise").astype("float64")
if salary.isna().any() or not np.isfinite(salary.to_numpy()).all():
    raise ValueError("Target salary contains missing, infinite or non-finite values.")
if not salary.between(MIN_SALARY, MAX_SALARY).all():
    raise ValueError(
        f"Salary outside the declared ${MIN_SALARY:,}–${MAX_SALARY:,} window. "
        "Do not filter silently here; review the cleaning export."
    )

y = np.log(salary)
y.name = "log_annual_salary_usd"

print("=== TARGET ===")
print("Observations:", len(y))
print("Raw annual USD range:", salary.min(), "to", salary.max())
display(y.describe().to_frame())

=== TARGET ===
Observations: 386701
Raw annual USD range: 1000.0 to 1000000.0


,log_annual_salary_usd
count,386701.000000
mean,10.875020
std,1.048681
min,6.907755
25%,10.355073
50%,11.047296
75%,11.548949
max,13.815511


## 3. Keep sensitivity metadata **outside** the predictor matrix

- `comp_outlier_country` is **derived from compensation** and is not a legitimate standard predictor. We retain it only for the preplanned Phase 9 sensitivity analysis.
- `Region` is fixed geographic metadata for later train-only country pooling; it is not an additional model input.
- `Country_grouped` from the exported CSV was calculated using **full-dataset country counts**. We *do not use it* in the primary predictor matrix. The grouping threshold must be learned from training data after the split.

The salary outlier flag was also calculated using the full dataset. Keep it out of `X`; any later sensitivity analysis based on that label-derived flag should explicitly document how it defines the evaluation sample.

In [20]:
salary_outlier_flag = df["comp_outlier_country"].copy()
region_metadata = df["Region"].copy()

assert salary_outlier_flag.index.equals(y.index)
assert region_metadata.index.equals(y.index)

print("=== SENSITIVITY METADATA (NOT PREDICTORS) ===")
display(salary_outlier_flag.value_counts(dropna=False).rename_axis("Flag").to_frame("Rows"))

=== SENSITIVITY METADATA (NOT PREDICTORS) ===


,Rows
Flag,
0,378062
1,8639


## 4. Select exactly the approved 17 predictors

An **allowlist** is safer than dropping a hand-selected blacklist from all 33 columns: newly added compensation fields cannot accidentally enter the model. We exclude `PurchaseInfluence` because its source question varied materially by year. We also exclude the language-count derivative as a *modeling choice*, not because it is salary leakage.

The 17 features are:

| Numeric (5) | Categorical (12) |
|:--|:--|
| `survey_year`, `Age`, `YearsCode`, `YearsCodePro`, `WorkExp` | `Country`, `OrgSize`, `EdLevel`, `Employment`, `Industry`, `RemoteWork`, `LearnCode`, `OpSys`, `MainBranch`, `DevType`, `LanguageHaveWorkedWith`, `LanguageWantToWorkWith` |

`OrgSize` belongs in the **categorical** list: numeric codes in the CSV represent survey bins, not measured employee counts.

In [21]:
NUMERIC = ["survey_year", "Age", "YearsCode", "YearsCodePro", "WorkExp"]

NOT_ASKED = {
    "EdLevel": list(range(2011, 2017)),
    "Employment": list(range(2011, 2015)),
    "Industry": list(range(2017, 2023)),
    "RemoteWork": [2011, 2012, 2013, 2018, 2020, 2021],
    "LearnCode": [2011, 2012, 2013, 2014, 2020],
    "OpSys": [2017, 2026],
    "MainBranch": list(range(2011, 2017)) + [2018, 2026],
    "LanguageWantToWorkWith": [2011, 2012, 2013],
    "OrgSize": [2014, 2015],
}
ALWAYS_ASKED = ["DevType", "LanguageHaveWorkedWith", "Country"]
CATEGORICAL = list(NOT_ASKED) + ALWAYS_ASKED
FEATURES = NUMERIC + CATEGORICAL

assert len(FEATURES) == 17 and len(set(FEATURES)) == 17
missing_features = sorted(set(FEATURES) - set(df.columns))
if missing_features:
    raise KeyError(f"Selected features missing from CSV: {missing_features}")

X = df.loc[:, FEATURES].copy()  # df and saved export remain unchanged
print("=== FEATURE SELECTION ===")
print("X:", X.shape, "| y:", y.shape)
print("Excluded CSV columns:", sorted(set(df.columns) - set(FEATURES)))

=== FEATURE SELECTION ===
X: (386701, 17) | y: (386701,)
Excluded CSV columns: ['CompFreq', 'CompIsTotalComp', 'CompTotal', 'Comp_USD_Annual', 'ConvertedCompYearly', 'Country_grouped', 'Currency', 'Currency_ISO', 'JobSat', 'LanguageHaveWorkedWith_count', 'PurchaseInfluence', 'Region', 'ResponseId', 'comp_outlier_country', 'comp_source', 'uid']


## 5. Verify numeric data — do not impute

Cast numeric features using `errors="raise"` so unexpected text is surfaced rather than turned into missing values. Leave all numeric nulls intact. In particular, `WorkExp` was **not asked before 2022**, and `YearsCodePro` is structurally absent in several years; their large missing counts must not be interpreted as ordinary skipped responses.

In [22]:
X["survey_year"] = pd.to_numeric(X["survey_year"], errors="raise").astype("int64")
for col in NUMERIC:
    if col != "survey_year":
        X[col] = pd.to_numeric(X[col], errors="raise")

numeric_missing_before = df[NUMERIC].isna().sum().astype(int)
numeric_missing_after = X[NUMERIC].isna().sum().astype(int)
assert numeric_missing_before.equals(numeric_missing_after), "Numeric missingness unexpectedly changed."

print("=== NUMERIC NULLS PRESERVED ===")
display(numeric_missing_after.rename("Missing").to_frame())

=== NUMERIC NULLS PRESERVED ===


,Missing
survey_year,0
Age,21590
YearsCode,1220
YearsCodePro,100098
WorkExp,280225


## 6. Decode `OrgSize` as categorical bins

The v2 export contains numeric *representatives* of organization-size categories. We map these exact seven codes back to their category labels. Unknown nonmissing codes raise an error rather than being silently assigned.

| CSV code | Category |
|--:|:--|
| 10 | Under 20 |
| 60 | 20–99 |
| 300 | 100–499 |
| 750 | 500–999 |
| 3,000 | 1,000–4,999 |
| 7,500 | 5,000–9,999 |
| 15,000 | 10,000+ |

**Limitation:** This only decodes the *exported* bin. It cannot recover respondents whose broad historical answers were placed into the wrong bin upstream (for example, `Fortune 1000 (1,000+)` is not evidence of 10,000+ employees). Correcting those cases requires the raw cleaning notebook and re-export, not guessing in Phase 3.

In [23]:
ORG_SIZE_MAP = {
    10: "Under 20",
    60: "20-99",
    300: "100-499",
    750: "500-999",
    3000: "1,000-4,999",
    7500: "5,000-9,999",
    15000: "10,000+",
}
org_raw = pd.to_numeric(X["OrgSize"], errors="raise")
unknown_org_codes = org_raw.notna() & ~org_raw.isin(ORG_SIZE_MAP)
if unknown_org_codes.any():
    raise ValueError(
        "Unrecognized OrgSize code(s): "
        f"{sorted(org_raw.loc[unknown_org_codes].unique().tolist())}. "
        "Review the upstream cleaning mapping."
    )
X["OrgSize"] = org_raw.map(ORG_SIZE_MAP).astype("string")

print("=== ORGSIZE CATEGORIES (BEFORE MISSINGNESS LABELS) ===")
display(X["OrgSize"].value_counts(dropna=False).to_frame("Rows"))

=== ORGSIZE CATEGORIES (BEFORE MISSINGNESS LABELS) ===


,Rows
OrgSize,
Under 20,80575
20-99,78942
100-499,69136
"10,000+",49623
"1,000-4,999",39839
<NA>,30075
500-999,23778
"5,000-9,999",14733


## 7. Label missing categorical answers with **fixed** year rules

- **Not asked:** The question was unavailable in that survey year and the corresponding exported value is blank.
- **No answer:** The question existed that year but this respondent's value is blank.
- **Literal responses** such as `None`, `NA`, or an actual respondent answer are not replaced.
- `OrgSize` is treated here alongside the other categorical fields, including its **2014–2015 Not asked** rule.

These labels are fixed questionnaire metadata, not fitted from feature distributions, and may therefore be created **before** the train/test split. They do not yet create one-hot columns; that happens later on training data.

In [24]:
for col in CATEGORICAL:
    values = X[col].astype("string").str.strip().replace("", pd.NA)
    if col in NOT_ASKED:
        missing_not_asked = values.isna() & X["survey_year"].isin(NOT_ASKED[col])
        values = values.mask(missing_not_asked, "Not asked")
    X[col] = values.fillna("No answer")

missingness_summary = pd.DataFrame({
    "Not asked": [int(X[c].eq("Not asked").sum()) for c in CATEGORICAL],
    "No answer": [int(X[c].eq("No answer").sum()) for c in CATEGORICAL],
    "Remaining null": [int(X[c].isna().sum()) for c in CATEGORICAL],
}, index=CATEGORICAL)

print("=== CATEGORICAL MISSINGNESS ===")
display(missingness_summary)
print("\n=== FINAL ORGANIZATION SIZE DISTRIBUTION ===")
display(X["OrgSize"].value_counts(dropna=False).to_frame("Rows"))

=== CATEGORICAL MISSINGNESS ===


,Not asked,No answer,Remaining null
EdLevel,65070,1870,0
Employment,14915,1375,0
Industry,219316,29108,0
RemoteWork,130531,7246,0
LearnCode,48911,23044,0
OpSys,24730,13587,0
MainBranch,118788,0,0
LanguageWantToWorkWith,10535,33686,0
OrgSize,17872,12203,0
DevType,0,2696,0



=== FINAL ORGANIZATION SIZE DISTRIBUTION ===


,Rows
OrgSize,
Under 20,80575
20-99,78942
100-499,69136
"10,000+",49623
"1,000-4,999",39839
500-999,23778
Not asked,17872
"5,000-9,999",14733
No answer,12203


## 8. Acceptance checks — all must pass

These assertions test the **actual Phase 3 contract**, not simply the absence of exceptions. They check target integrity, row/index alignment, categorical labeling rules for blank values, numeric preservation, known leakage fields, expected column roles, and retained sensitivity metadata.

In [25]:
# Target / row alignment
assert len(X) == len(y) == len(df)
assert X.index.equals(y.index) and X.columns.is_unique
assert np.isfinite(y.to_numpy()).all()
np.testing.assert_allclose(y.to_numpy(), np.log(salary.to_numpy()), rtol=0, atol=1e-12)
assert salary.between(MIN_SALARY, MAX_SALARY).all()

# Exact schema and data types
assert set(X.columns) == set(FEATURES) == set(NUMERIC + CATEGORICAL)
assert len(NUMERIC) == 5 and len(CATEGORICAL) == 12
assert pd.api.types.is_integer_dtype(X["survey_year"])
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in NUMERIC)
assert "OrgSize" in CATEGORICAL and "OrgSize" not in NUMERIC
assert X[CATEGORICAL].notna().all().all()
assert X[NUMERIC].isna().sum().astype(int).equals(numeric_missing_before)

# Leakage, IDs, metadata and survey incompatibilities excluded
FORBIDDEN = {
    TARGET, "CompTotal", "ConvertedCompYearly", "comp_outlier_country",
    "Currency", "Currency_ISO", "CompFreq", "comp_source", "CompIsTotalComp",
    "uid", "ResponseId", "JobSat", "Region", "Country_grouped",
    "PurchaseInfluence", "LanguageHaveWorkedWith_count",
}
assert not (set(X.columns) & FORBIDDEN)
assert not any(c.startswith("PurchaseInfluence__") for c in X.columns)
assert "Country" in X.columns and "WorkExp" in X.columns
assert "comp_outlier_country" in df.columns
assert salary_outlier_flag.index.equals(y.index)
assert region_metadata.index.equals(y.index)

# Every blank raw categorical response receives the appropriate fixed label.
for col in CATEGORICAL:
    raw = df[col].astype("string").str.strip()
    missing_raw = raw.isna() | raw.eq("").fillna(False)
    not_asked_year = df["survey_year"].isin(NOT_ASKED.get(col, []))
    expected_not_asked = missing_raw & not_asked_year
    expected_no_answer = missing_raw & ~not_asked_year
    assert X.loc[expected_not_asked, col].eq("Not asked").all(), col
    assert X.loc[expected_no_answer, col].eq("No answer").all(), col

print("PHASE 3: ALL ACCEPTANCE CHECKS PASSED")
print(f"Rows: {len(X):,} | predictors: {len(FEATURES)} ({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print("Target: log annual USD salary | survey_year dtype:", X["survey_year"].dtype)
print("Original df and CSV were not otherwise modified.")

PHASE 3: ALL ACCEPTANCE CHECKS PASSED
Rows: 386,701 | predictors: 17 (5 numeric, 12 categorical)
Target: log annual USD salary | survey_year dtype: int64
Original df and CSV were not otherwise modified.


## 9. Handoff summary for Phase 4 (inspection only)

Here we print the per-year sample sizes and missing numeric counts. No preprocessing is fitted. This is especially important for `WorkExp` and `YearsCodePro`, whose absence is partly **structural**. Keep survey-year stratification, missingness indicators, or feature-subset comparisons in mind when designing the **training-only** imputation strategy.

In [26]:
print("=== MODELING ROWS BY SURVEY YEAR ===")
display(X.groupby("survey_year").size().rename("Respondents").to_frame())

print("=== NUMERIC MISSINGNESS BY YEAR (PERCENT) ===")
display((X[NUMERIC].isna().groupby(X["survey_year"]).mean() * 100).round(1))

print("=== READY FOR SPLIT ===")
print("X:", X.shape, "y:", y.shape)
print("Outlier flag:", salary_outlier_flag.shape, "region metadata:", region_metadata.shape)

=== MODELING ROWS BY SURVEY YEAR ===


,Respondents
survey_year,
2011,2031
2012,3466
2013,5038
2014,4380
2015,13492
2016,36663
2017,12573
2018,41561
2019,50402


=== NUMERIC MISSINGNESS BY YEAR (PERCENT) ===


,survey_year,Age,YearsCode,YearsCodePro,WorkExp
survey_year,,,,,
2011,0.0,0.0,0.0,100.0,100.0
2012,0.0,0.0,0.0,100.0,100.0
2013,0.0,0.0,0.0,100.0,100.0
2014,0.0,0.0,0.0,100.0,100.0
2015,0.0,0.6,0.1,100.0,100.0
2016,0.0,0.4,0.4,100.0,100.0
2017,0.0,100.0,0.1,0.4,100.0
2018,0.0,5.6,0.0,0.3,100.0
2019,0.0,4.2,0.2,0.3,100.0


=== READY FOR SPLIT ===
X: (386701, 17) y: (386701,)
Outlier flag: (386701,) region metadata: (386701,)


## 10. Known upstream limitations and next-phase rules

**Passed Phase 3 ≠ validated raw survey harmonization.** The following issues cannot be fully repaired from the 33-column export and should be addressed in the cleaning source before treating related coefficients as final:

1. **Organization-size misclassification:** Some very broad original sizes (e.g., `Fortune 1000 (1,000+)`, `3001+`) were assigned to narrow bins in the v2 cleaning pipeline. This notebook correctly treats resulting bins as categories but cannot restore the lost ranges.
2. **Historical ages:** Some range-to-bracket approximations in v2 do not preserve the original age interval exactly. Original response data is needed for corrected ages.
3. **Employment and remote work:** Generic `Employed` responses and occasionally-remote categories were mapped to more specific values upstream. The exact original categories are no longer available in this export.
4. **Salary and population provenance:** The v2 export favors published `ConvertedCompYearly` and applies its own respondent exclusions. It is **not** interchangeable with the separately audited salary-cleaning version.
5. **Country-year salary outlier flag:** Computed from salary and the full exported dataset. It is preserved for sensitivity work but excluded from predictors. Do not use it to choose features or train/test preprocessing.

**Phase 4+ requirements:** Split respondents first; fit country-count pooling on *training only* and apply the learned mapping to test; impute and scale numeric fields using training data; multi-hot encode semicolon-delimited fields instead of treating combinations as categories; explicitly route `survey_year` through the numeric transformer; make any salary-based sensitivity analysis a documented separate evaluation.

**Reproducibility step:** Run **Kernel → Restart & Run All**, then **Save** the notebook. If the export is unchanged, the counts should match the reference results above. If they differ, inspect the cleaning notebook's provenance rather than forcibly adjusting the data to make the numbers match.